# **Phase 2 — Modeling**

# **Imports**

In [ ]:
import pandas as pd
import numpy as np
import re
import time
import ast
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# **Load Processed Data**

In [ ]:
df = pd.read_csv('Processed_data.csv')

# Parse entities column from string to list
df['entities'] = df['entities'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else [])

df.head()

# **Train / Validation / Test Split**

In [ ]:
X = df["Text"]
y = df["label"]

# 70% Train
# 15% Validation
# 15% Test

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print(f"Train : {len(X_train)}")
print(f"Val   : {len(X_val)}")
print(f"Test  : {len(X_test)}")

# **Evaluation Helper**

In [ ]:
def evaluate(
    name,
    y_true_train,
    y_pred_train,
    y_true_val,
    y_pred_val,
    y_true_test,
    y_pred_test,
    elapsed
):

    def metrics(y_true, y_pred):
        return {
            "acc": accuracy_score(y_true, y_pred),
            "p": precision_score(y_true, y_pred, zero_division=0),
            "r": recall_score(y_true, y_pred, zero_division=0),
            "f1": f1_score(y_true, y_pred, zero_division=0)
        }

    train = metrics(y_true_train, y_pred_train)
    val = metrics(y_true_val, y_pred_val)
    test = metrics(y_true_test, y_pred_test)

    ms = elapsed * 1000 / len(y_true_test)

    print(f"\n── {name} ──")
    print(f' {"Split":<6} {"Acc":>7} {"Precision":>10} {"Recall":>8} {"F1":>8}')
    print(f' {"─"*43}')

    print(
        f' {"Train":<6} '
        f'{train["acc"]:>7.4f} '
        f'{train["p"]:>10.4f} '
        f'{train["r"]:>8.4f} '
        f'{train["f1"]:>8.4f}'
    )

    print(
        f' {"Val":<6} '
        f'{val["acc"]:>7.4f} '
        f'{val["p"]:>10.4f} '
        f'{val["r"]:>8.4f} '
        f'{val["f1"]:>8.4f}'
    )

    print(
        f' {"Test":<6} '
        f'{test["acc"]:>7.4f} '
        f'{test["p"]:>10.4f} '
        f'{test["r"]:>8.4f} '
        f'{test["f1"]:>8.4f}'
    )

    print(f" Inference : {ms:.2f} ms/sample")

    return {
        "model": name,
        "test_acc": round(test["acc"], 4),
        "test_precision": round(test["p"], 4),
        "test_recall": round(test["r"], 4),
        "test_f1": round(test["f1"], 4),
        "ms_per_sample": round(ms, 2)
    }

# **Approach 1 — Classical ML (TF-IDF + SVM)**

In [ ]:
# ===================================================
#  Imports
# ===================================================

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline

In [ ]:
svm_pipeline = Pipeline([

    (
        "tfidf",
        TfidfVectorizer(
            ngram_range=(1, 3),
            max_features=30000,
            min_df=2,
            max_df=0.95,
            sublinear_tf=True )
    ),

    (
        "clf",
        LinearSVC(
            C=0.1,
            max_iter=3000 )
    )

])

In [ ]:
svm_pipeline.fit(
    X_train,
    y_train
)

In [ ]:
y_pred_train_svm = svm_pipeline.predict(
    X_train
)

y_pred_val_svm = svm_pipeline.predict(
    X_val
)

start = time.time()

y_pred_test_svm = svm_pipeline.predict(
    X_test
)

elapsed_svm = time.time() - start

In [ ]:
results_svm = evaluate(
    "Classical ML — TF-IDF + SVM",

    y_train,
    y_pred_train_svm,

    y_val,
    y_pred_val_svm,

    y_test,
    y_pred_test_svm,

    elapsed_svm
)

In [ ]:
cm = confusion_matrix(
    y_test,
    y_pred_test_svm
)

print(cm)

In [ ]:
results_svm

# **Approach 2 — Deep Learning (BiLSTM)**

In [ ]:
# ===================================================
#  Imports
# ===================================================

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
MAX_WORDS = 20000
MAX_LEN = 100

tokenizer = Tokenizer(
    num_words=MAX_WORDS,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(X_train)

## Text → Sequences

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq = tokenizer.texts_to_sequences(X_val)
X_test_seq = tokenizer.texts_to_sequences(X_test)

In [ ]:
X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

X_val_pad = pad_sequences(
    X_val_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

In [ ]:
# ===================================================
# Download FastText Arabic Embeddings (Run Once)
# ===================================================

import os

if not os.path.exists("cc.ar.300.vec"):

    !wget -q https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.ar.300.vec.gz

    !gunzip -f cc.ar.300.vec.gz

print(
    os.path.exists("cc.ar.300.vec")
)

In [ ]:
EMBEDDING_DIM = 300

embeddings_index = {}

with open(
    "cc.ar.300.vec",
    encoding="utf-8"
) as f:

    next(f)

    for line in f:

        values = line.rstrip().split(" ")

        word = values[0]

        vector = np.asarray(
            values[1:],
            dtype="float32"
        )

        if len(vector) == EMBEDDING_DIM:
            embeddings_index[word] = vector

print(
    "Loaded",
    len(embeddings_index),
    "word vectors"
)

In [ ]:
embedding_matrix = np.zeros(
    (MAX_WORDS, EMBEDDING_DIM)
)

word_index = tokenizer.word_index

hits = 0
misses = 0

for word, idx in word_index.items():

    if idx >= MAX_WORDS:
        continue

    vector = embeddings_index.get(word)

    if vector is not None:

        embedding_matrix[idx] = vector
        hits += 1

    else:

        misses += 1

print("Found :", hits)
print("Missed:", misses)

coverage = hits / (hits + misses)

print(
    f"Coverage = {coverage:.2%}"
)

In [ ]:
# ===================================================
#  Imports
# ===================================================

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import (
    Embedding,
    Bidirectional,
    LSTM,
    Dense,
    Dropout
)

In [ ]:
bilstm_model = Sequential([

    Embedding(
        input_dim=MAX_WORDS,
        output_dim=EMBEDDING_DIM,
        weights=[embedding_matrix],
        trainable=False
    ),

    Bidirectional(
        LSTM(
            128,
            return_sequences=True,
            dropout=0.2
        )
    ),

    Bidirectional(
        LSTM(
            64,
            dropout=0.2
        )
    ),

    Dense(
        64,
        activation="relu"
    ),

    Dropout(0.5),

    Dense(
        1,
        activation="sigmoid"
    )

])

In [ ]:
bilstm_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

bilstm_model.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

In [ ]:
print("model" in globals())

In [ ]:
## Step 9 — Compile

bilstm_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

bilstm_model.summary()

In [ ]:
# =======================
# Imports
# =======================
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

In [ ]:
history = bilstm_model.fit(

    X_train_pad,
    y_train,

    validation_data=(
        X_val_pad,
        y_val
    ),

    epochs=15,
    batch_size=16,

    callbacks=[early_stop],

    verbose=1
)

In [ ]:
y_pred_train_bilstm = (
    bilstm_model.predict(
        X_train_pad,
        verbose=0
    ) > 0.5
).astype(int)

y_pred_val_bilstm = (
    bilstm_model.predict(
        X_val_pad,
        verbose=0
    ) > 0.5
).astype(int)

start = time.time()

y_pred_test_bilstm = (
    bilstm_model.predict(
        X_test_pad,
        verbose=0
    ) > 0.5
).astype(int)

elapsed_bilstm = (
    time.time() - start
)

In [ ]:
results_bilstm = evaluate(

    "Deep Learning — BiLSTM + FastText",

    y_train,
    y_pred_train_bilstm,

    y_val,
    y_pred_val_bilstm,

    y_test,
    y_pred_test_bilstm,

    elapsed_bilstm
)

In [ ]:
cm = confusion_matrix(
    y_test,
    y_pred_test_bilstm
)

print(cm)

In [ ]:
results_bilstm

# **Approach 3 — Transformer (AraBERT Fine-tuning)**

In [ ]:
!pip install transformers -q

In [ ]:
import torch
import numpy as np
import time

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

from torch.utils.data import (
    Dataset,
    DataLoader
)

from torch.optim import AdamW

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

from tqdm import tqdm

In [ ]:
MODEL_NAME = "aubmindlab/bert-base-arabertv02"

hf_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

arabert_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

In [ ]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

arabert_model = arabert_model.to(device)

print(device)

In [ ]:
class ArabicDataset(Dataset):

    def __init__(
        self,
        texts,
        labels,
        tokenizer,
        max_len
    ):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        text = str(self.texts[idx])

        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        )

        return {
            "input_ids":
                encoding["input_ids"].squeeze(),

            "attention_mask":
                encoding["attention_mask"].squeeze(),

            "labels":
                torch.tensor(
                    self.labels[idx],
                    dtype=torch.long
                )
        }

In [ ]:
MAX_LEN = 128

train_dataset = ArabicDataset(
    X_train,
    y_train,
    hf_tokenizer,
    MAX_LEN
)

val_dataset = ArabicDataset(
    X_val,
    y_val,
    hf_tokenizer,
    MAX_LEN
)

test_dataset = ArabicDataset(
    X_test,
    y_test,
    hf_tokenizer,
    MAX_LEN
)

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16
)

In [ ]:
optimizer = AdamW(
    arabert_model.parameters(),
    lr=2e-5
)

In [ ]:
def train_epoch(model, loader):

    model.train()

    total_loss = 0

    for batch in tqdm(loader):

        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)

        attention_mask = batch["attention_mask"].to(device)

        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
def evaluate_epoch(model, loader):

    model.eval()

    predictions = []
    true_labels = []

    with torch.no_grad():

        for batch in loader:

            input_ids = batch["input_ids"].to(device)

            attention_mask = batch["attention_mask"].to(device)

            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            preds = torch.argmax(
                outputs.logits,
                dim=1
            )

            predictions.extend(
                preds.cpu().numpy()
            )

            true_labels.extend(
                labels.cpu().numpy()
            )

    acc = accuracy_score(
        true_labels,
        predictions
    )

    p = precision_score(
        true_labels,
        predictions
    )

    r = recall_score(
        true_labels,
        predictions
    )

    f1 = f1_score(
        true_labels,
        predictions
    )

    return acc, p, r, f1

In [ ]:
EPOCHS = 2

for epoch in range(EPOCHS):

    train_loss = train_epoch(
        arabert_model,
        train_loader
    )

    val_acc, val_p, val_r, val_f1 = evaluate_epoch(
        arabert_model,
        val_loader
    )

    print()
    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Acc : {val_acc:.4f}")
    print(f"Val F1  : {val_f1:.4f}")

In [ ]:
start = time.time()

test_acc, test_p, test_r, test_f1 = evaluate_epoch(
    arabert_model,
    test_loader
)

elapsed = time.time() - start

ms = elapsed * 1000 / len(y_test)

print("\n── Transformer — AraBERT ──")

print(f"Test Acc       : {test_acc:.4f}")
print(f"Test Precision : {test_p:.4f}")
print(f"Test Recall    : {test_r:.4f}")
print(f"Test F1        : {test_f1:.4f}")
print(f"Inference      : {ms:.2f} ms/sample")

In [ ]:
arabert_model.eval()

predictions = []
true_labels = []

with torch.no_grad():

    for batch in test_loader:

        input_ids = batch["input_ids"].to(device)

        attention_mask = batch["attention_mask"].to(device)

        outputs = arabert_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        preds = torch.argmax(
            outputs.logits,
            dim=1
        )

        predictions.extend(
            preds.cpu().numpy()
        )

        true_labels.extend(
            batch["labels"].numpy()
        )

cm = confusion_matrix(
    true_labels,
    predictions
)

print(cm)

In [ ]:
results_arabert = {
    "model": "Transformer — AraBERT",
    "test_acc": round(test_acc, 4),
    "test_precision": round(test_p, 4),
    "test_recall": round(test_r, 4),
    "test_f1": round(test_f1, 4),
    "ms_per_sample": round(ms, 2)
}

results_arabert

# **Approach 4 — LLM**

In [ ]:
!pip install openai -q

import time

from openai import OpenAI

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

In [ ]:
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key="YOUR_API_KEY"
)

In [ ]:
def build_prompt(tweet):

    return f"""
You are an expert Arabic Comparative Opinion Mining classifier.

Task:
Classify the tweet.

Return ONLY:
1
or
0

Label = 1 ONLY IF:
- The author expresses a comparative opinion.
- The author clearly prefers one entity over another.
- There is a clear winner and loser.

Label = 0 IF:
- It is a question.
- It asks for advice.
- It asks which is better.
- It discusses a comparison.
- It reports someone else's comparison.
- It mentions multiple brands without preference.
- It is news or information.
- No clear comparative opinion from the author.

Examples:

Tweet:
سامسونج افضل من ابل بمراحل
Answer:
1

Tweet:
ايفون افضل من اي هاتف اندرويد
Answer:
1

Tweet:
سوني تتفوق على مايكروسوفت
Answer:
1

Tweet:
اخ محمد تنصحني بالابتوب ولا الايباد افضل؟
Answer:
0

Tweet:
هل ايفون افضل من جالكسي؟
Answer:
0

Tweet:
انت تريد تقنعنا ان ايفون افضل من جالكسي
Answer:
0

Tweet:
اتش بي افضل من سامسونج وبكم سعرها؟
Answer:
0

Tweet:
ابل اكبر مستهلك لخدمات جوجل السحابية
Answer:
0

Tweet:
سامسونج وابل شركتان كبيرتان
Answer:
0

Tweet:
ودي اشوف ليه الناس تستخدم ايفون اكثر من هواوي
Answer:
0

Tweet:
{tweet}

Answer:
"""

In [ ]:
def predict_qwen(tweet):

    prompt = build_prompt(tweet)

    response = client.chat.completions.create(
        model="qwen/qwen-2.5-7b-instruct",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0,
        max_tokens=3
    )

    return response.choices[0].message.content.strip()

In [ ]:
sample = X_test.iloc[0]

print(sample)

print(
    predict_qwen(sample)
)

In [ ]:
predictions = []

start = time.time()

for text in X_test:

    pred = predict_qwen(text)

    pred = 1 if str(pred).startswith("1") else 0

    predictions.append(pred)

elapsed = time.time() - start

In [ ]:
acc = accuracy_score(
    y_test,
    predictions
)

p = precision_score(
    y_test,
    predictions
)

r = recall_score(
    y_test,
    predictions
)

f1 = f1_score(
    y_test,
    predictions
)

ms = elapsed * 1000 / len(y_test)

In [ ]:
print("\n── LLM — Qwen Few-Shot ──")

print(f"Accuracy  : {acc:.4f}")
print(f"Precision : {p:.4f}")
print(f"Recall    : {r:.4f}")
print(f"F1        : {f1:.4f}")
print(f"Inference : {ms:.2f} ms/sample")

In [ ]:
cm = confusion_matrix(
    y_test,
    predictions
)

print(cm)

In [ ]:
# **Approach 4 — LLM**

In [ ]:
results_qwen = {
    "model": "LLM — Qwen Few-Shot",
    "test_acc": round(acc, 4),
    "test_precision": round(p, 4),
    "test_recall": round(r, 4),
    "test_f1": round(f1, 4),
    "ms_per_sample": round(ms, 2)
}

results_qwen

# **Phase 3 — Evaluation**

In [ ]:
comparison = pd.DataFrame([
    results_svm,
    results_bilstm,
    results_arabert,
    results_qwen
])

comparison

In [ ]:
ner_df = df.copy()

ner_df = ner_df[
    ner_df["entities"].apply(len) > 0
].reset_index(drop=True)

print("NER Samples:", len(ner_df))

ner_df[
    ["Text", "entities"]
].head()

In [ ]:
def text_to_bio(text, entities):

    tokens = text.split()

    labels = ["O"] * len(tokens)

    for entity in entities:

        entity_tokens = str(entity).split()

        n = len(entity_tokens)

        for i in range(len(tokens) - n + 1):

            if tokens[i:i+n] == entity_tokens:

                labels[i] = "B-BRAND"

                for j in range(1, n):

                    labels[i+j] = "I-BRAND"

    return tokens, labels


ner_tokens = []
ner_labels = []

for _, row in ner_df.iterrows():

    tokens, labels = text_to_bio(
        row["Text"],
        row["entities"]
    )

    ner_tokens.append(tokens)
    ner_labels.append(labels)

In [ ]:
label_list = [
    "O",
    "B-BRAND",
    "I-BRAND"
]

label2id = {
    label: idx
    for idx, label in enumerate(label_list)
}

id2label = {
    idx: label
    for label, idx in label2id.items()
}

print(label2id)

In [ ]:
from sklearn.model_selection import train_test_split

X_train_tokens, X_temp_tokens, y_train_labels, y_temp_labels = train_test_split(
    ner_tokens,
    ner_labels,
    test_size=0.30,
    random_state=42
)

X_val_tokens, X_test_tokens, y_val_labels, y_test_labels = train_test_split(
    X_temp_tokens,
    y_temp_labels,
    test_size=0.50,
    random_state=42
)

print("Train:", len(X_train_tokens))
print("Val  :", len(X_val_tokens))
print("Test :", len(X_test_tokens))

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification
)

MODEL_NAME = "aubmindlab/bert-base-arabertv02"

hf_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

ner_model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

In [ ]:
from torch.utils.data import Dataset

class NERDataset(Dataset):

    def __init__(
        self,
        tokens,
        labels,
        tokenizer,
        max_len=128
    ):
        self.tokens = tokens
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.tokens)

    def __getitem__(self, idx):

        words = self.tokens[idx]

        word_labels = [
            label2id[x]
            for x in self.labels[idx]
        ]

        encoding = self.tokenizer(
            words,
            is_split_into_words=True,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        word_ids = encoding.word_ids()

        label_ids = []

        previous_word = None

        for word_idx in word_ids:

            if word_idx is None:

                label_ids.append(-100)

            elif word_idx != previous_word:

                label_ids.append(
                    word_labels[word_idx]
                )

            else:

                label_ids.append(
                    word_labels[word_idx]
                )

            previous_word = word_idx

        return {
            "input_ids":
                encoding["input_ids"].squeeze(),

            "attention_mask":
                encoding["attention_mask"].squeeze(),

            "labels":
                torch.tensor(label_ids)
        }

In [ ]:
from torch.utils.data import DataLoader

train_dataset = NERDataset(
    X_train_tokens,
    y_train_labels,
    hf_tokenizer
)

val_dataset = NERDataset(
    X_val_tokens,
    y_val_labels,
    hf_tokenizer
)

test_dataset = NERDataset(
    X_test_tokens,
    y_test_labels,
    hf_tokenizer
)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16
)

In [ ]:
import torch
from torch.optim import AdamW

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

ner_model = ner_model.to(device)

optimizer = AdamW(
    ner_model.parameters(),
    lr=2e-5
)

print(device)

In [ ]:
from tqdm import tqdm

def train_epoch(model, loader):

    model.train()

    total_loss = 0

    for batch in tqdm(loader):

        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
def evaluate_token_accuracy(
    model,
    loader
):

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for batch in loader:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            preds = torch.argmax(
                outputs.logits,
                dim=2
            )

            mask = labels != -100

            correct += (
                (preds == labels)[mask]
            ).sum().item()

            total += mask.sum().item()

    return correct / total

In [ ]:
EPOCHS = 3

for epoch in range(EPOCHS):

    train_loss = train_epoch(
        ner_model,
        train_loader
    )

    val_acc = evaluate_token_accuracy(
        ner_model,
        val_loader
    )

    print()
    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"Loss      : {train_loss:.4f}")
    print(f"Token Acc : {val_acc:.4f}")

In [ ]:
!pip install seqeval -q

from seqeval.metrics import (
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

true_predictions = []
true_labels = []

ner_model.eval()

with torch.no_grad():

    for batch in test_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = ner_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        preds = torch.argmax(
            outputs.logits,
            dim=2
        )

        preds = preds.cpu().numpy()
        labels = labels.cpu().numpy()

        for pred, label in zip(preds, labels):

            pred_tags = []
            true_tags = []

            for p, l in zip(pred, label):

                if l != -100:

                    pred_tags.append(
                        id2label[p]
                    )

                    true_tags.append(
                        id2label[l]
                    )

            true_predictions.append(pred_tags)
            true_labels.append(true_tags)

In [ ]:
ner_precision = precision_score(
    true_labels,
    true_predictions
)

ner_recall = recall_score(
    true_labels,
    true_predictions
)

ner_f1 = f1_score(
    true_labels,
    true_predictions
)

print("\n── NER Evaluation ──")
print(f"Precision : {ner_precision:.4f}")
print(f"Recall    : {ner_recall:.4f}")
print(f"F1 Score  : {ner_f1:.4f}")

print(
    classification_report(
        true_labels,
        true_predictions
    )
)

# **Phase 4 —  Engineering & Deployment**

In [ ]:
arabert_model.save_pretrained(
    "comparison_model"
)

hf_tokenizer.save_pretrained(
    "comparison_model"
)


ner_model.save_pretrained(
    "ner_model"
)

hf_tokenizer.save_pretrained(
    "ner_model"
)

In [ ]:
import os

print(
    os.listdir("comparison_model")
)

print(
    os.listdir("ner_model")
)

In [ ]:
!pip install fastapi uvicorn transformers torch -q

In [ ]:
!nohup uvicorn app:app \
    --host 0.0.0.0 \
    --port 8000 \
    > server.log 2>&1 &

In [ ]:
import requests

response = requests.get(
    "http://127.0.0.1:8000"
)

print(
    response.json()
)

## API Testing

After deploying the FastAPI application, the following tests were performed to verify that the API was functioning correctly.

In [ ]:
import requests

response = requests.get(
    "http://127.0.0.1:8000"
)

print("Status Code:", response.status_code)
print(response.json())

In [ ]:
import requests

response = requests.post(
    "http://127.0.0.1:8000/predict",
    json={
        "text": "سامسونج أفضل من ايفون بمراحل"
    }
)

print("Status Code:", response.status_code)
print(response.json())

In [ ]:
import requests

response = requests.post(
    "http://127.0.0.1:8000/predict",
    json={
        "text": "الهاتف جميل جدا"
    }
)

print(response.json())

## Deployment Verification

The API was successfully deployed using FastAPI and tested through HTTP requests. Both the health check endpoint and prediction endpoint returned valid responses, confirming successful deployment and model integration.